# 01 · Signals

Distributions and correlations of the year-over-year change signals (the paper's Table I, Panels B and C). Build the inputs with `filinglens research signals`. All logic lives in `filinglens.research`; this notebook only looks at the results.

In [ ]:
import pandas as pd

from filinglens.config import get_settings
from filinglens.research.prices import add_characteristics, load_panel
from filinglens.research.signals import load_signals
from filinglens.research.synthetic import make_world

RESEARCH = get_settings().processed_dir / "research"
REAL = all(
    (RESEARCH / f).exists() for f in ("signals.parquet", "prices.parquet", "factors.parquet")
)
if REAL:
    signals = load_signals(RESEARCH / "signals.parquet")
    panel = add_characteristics(load_panel(RESEARCH / "prices.parquet"))
    factors = pd.read_parquet(RESEARCH / "factors.parquet")
    factors.index = pd.PeriodIndex(factors.pop("month"), freq="M")
else:
    print("No research data yet: using the SYNTHETIC world (pipeline check, not results).")
    world = make_world()
    signals, panel, factors = world.signals, add_characteristics(world.panel), world.factors
source = panel.attrs.get("source")
print(f"{signals['cik'].nunique()} firms, {len(signals)} signal rows, returns: {source}")

In [ ]:
SIGS = [
    c
    for c in [
        "sim_cosine_tf",
        "sim_jaccard",
        "sim_minedit",
        "sim_simple",
        "sim_embed",
        "sim_embed_aligned",
        "n_new_risks",
        "lm_neg_delta",
    ]
    if c in signals and signals[c].notna().any()
]
full = signals[signals["section"] == "full"]
full[SIGS].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(3)

Correlations between measures. The paper reports that all four classical measures are strongly positively correlated, with Sim_Simple the least related to Sim_Cosine.

In [ ]:
full[SIGS].corr().round(2)

In [ ]:
if "words_new" in signals:
    by_year = full.assign(year=full["filing_month"].dt.year).groupby("year")["words_new"].median()
    ax = by_year.plot(
        title="Median 10-K length (words)"
    )  # the paper: 10-Ks were 6x longer in 2014 than 1995
    ax.set_ylabel("words")

In [ ]:
if "section" in signals:
    signals.groupby("section")[SIGS].mean().round(3)